In [3]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys


def ensure_package(package_name, import_name=None):
    target = import_name or package_name
    if importlib.util.find_spec(target) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


ensure_package("pyyaml", "yaml")

import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "ppg_eeg").is_dir():
    repo_root = repo_root.parent

if not (repo_root / "ppg_eeg").is_dir():
    raise RuntimeError("Could not locate repo root containing ppg_eeg.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ppg_eeg import plot_correlation_heatmap

inputs = [
    ("ds003838", "derivatives/run_ds003838_fast/ds003838/correlations_fdr.csv"),
    ("ds006848", "derivatives/run_ds006848_fast/ds006848/correlations_fdr.csv"),
]

for dataset_id, csv_path in inputs:
    df = pd.read_csv(csv_path)

    fig, ax = plot_correlation_heatmap(
        df,
        dataset_id=dataset_id,
        method="spearman",
        show_values=True,
    )
    fig.savefig(f"derivatives/{dataset_id}_correlation_heatmap.png", dpi=300, bbox_inches="tight")
    plt.close(fig)